# Exercise 0 — Environment Setup & Verification

**Goal:** Confirm that your Python environment has everything needed for the remaining exercises.

Run each cell from top to bottom. Every check should print a green **PASS**. If you see a red **FAIL**, follow the troubleshooting guidance printed with it before moving on.

### What gets checked
| # | Check | Why |
|---|-------|-----|
| 1 | Python version | We require Python 3.10+ |
| 2 | Core libraries | `numpy`, `scipy`, `matplotlib` |
| 3 | Control systems library | `control` (python-control) |
| 4 | Visualization & interactivity | `plotly`, `ipywidgets`, `sympy`, `anywidget` |
| 5 | Quick smoke test | Build a small system and plot it to verify everything works end-to-end |

## Check 1 — Python Version

In [ ]:
import sys

major, minor = sys.version_info.major, sys.version_info.minor
if major >= 3 and minor >= 10:
    print(f"\033[92mPASS\033[0m  Python {major}.{minor}.{sys.version_info.micro}")
else:
    print(f"\033[91mFAIL\033[0m  Python {major}.{minor} detected — 3.10+ required")
    print("       Fix: create a new environment with Python 3.10+")
    print("         conda create -n control-demo python=3.10")
    print("         conda activate control-demo")

## Check 2 — Core Scientific Libraries

In [ ]:
def _check(pkg_name, import_name=None):
    """Try to import a package and report pass/fail with version."""
    import_name = import_name or pkg_name
    try:
        mod = __import__(import_name)
        ver = getattr(mod, "__version__", "unknown")
        print(f"\033[92mPASS\033[0m  {pkg_name} {ver}")
        return True
    except ImportError:
        print(f"\033[91mFAIL\033[0m  {pkg_name} not found")
        print(f"       Fix: pip install {pkg_name}")
        return False

all_ok = True
for pkg, imp in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    all_ok &= _check(pkg, imp)

## Check 3 — Control Systems Library

In [ ]:
_check("control", "control")

## Check 4 — Visualization & Interactivity

In [ ]:
for pkg, imp in [("plotly", "plotly"), ("ipywidgets", "ipywidgets"), ("sympy", "sympy"), ("anywidget", "anywidget")]:
    _check(pkg, imp)

## Check 5 — Smoke Test

If the cell below runs without error and shows an interactive plot, your environment is ready.

This builds a small coupled mass-spring-damper system and plots its step response using the same tools the later exercises rely on. Try moving the sliders to confirm widget interactivity works.

In [ ]:
import numpy as np
import control as ct
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

DEFAULT_M, DEFAULT_C, DEFAULT_K = 1.0, 0.1, 2.0

def build_system(m, c, k):
    A = np.array([
        [0,      0,      1,     0],
        [0,      0,      0,     1],
        [-2*k/m,  k/m,  -c/m,   0],
        [ k/m,  -2*k/m,  0,    -c/m]
    ])
    B = np.array([[0], [0], [0], [k/m]])
    C = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
    D = np.array([[0], [0]])
    return ct.ss(A, B, C, D)

def plot_response(response_type="step", m=DEFAULT_M, c=DEFAULT_C, k=DEFAULT_K):
    sys = build_system(m, c, k)
    T = np.linspace(0, 20, 400)
    if response_type == "initial":
        resp = ct.initial_response(sys, T=T, X0=[1, 0, 0, 0])
    elif response_type == "forced":
        resp = ct.forced_response(sys, T=T, U=np.ones_like(T))
    else:
        resp = ct.step_response(sys, T=T)
    t, y = resp.time, resp.outputs
    if y.ndim == 3:
        y = y[:, 0, :]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("q1 displacement", "q2 displacement"))
    fig.add_trace(go.Scatter(x=t, y=y[0], name="q1", mode="lines"), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=y[1], name="q2", mode="lines"), row=2, col=1)
    fig.update_layout(title=f"{response_type.title()} Response (m={m:.2f}, c={c:.2f}, k={k:.2f})",
                      template="plotly_white", height=500)
    fig.update_xaxes(title_text="Time (s)", row=2, col=1)
    fig.update_yaxes(title_text="q1", row=1, col=1)
    fig.update_yaxes(title_text="q2", row=2, col=1)
    display(fig)

ui = widgets.VBox([
    widgets.Dropdown(options=["step", "initial", "forced"], value="step", description="response"),
    widgets.FloatSlider(value=DEFAULT_M, min=0.5, max=5.0, step=0.1, description="m"),
    widgets.FloatSlider(value=DEFAULT_C, min=0.0, max=2.0, step=0.05, description="c"),
    widgets.FloatSlider(value=DEFAULT_K, min=0.5, max=10.0, step=0.1, description="k"),
])
out = widgets.interactive_output(plot_response, {
    "response_type": ui.children[0], "m": ui.children[1],
    "c": ui.children[2], "k": ui.children[3]
})
display(ui, out)
print("\n\033[92mPASS\033[0m  Smoke test complete — widgets and plotting are working!")

---

## All Checks Passed?

If every cell above shows **PASS**, you are ready to proceed to **Exercise 01 — System Dynamics**.

If any check failed, install the missing package and restart your kernel:

```bash
pip install -r requirements.txt
```

### Quick-Install (all dependencies)
```bash
pip install numpy scipy matplotlib control plotly ipywidgets sympy anywidget
```

### Exercise Roadmap

| Exercise | Folder | Topic |
|----------|--------|-------|
| 0 | `Exercise_00_Environment_Setup` | **You are here** — environment verification |
| 1 | `Exercise_01_System_Dynamics` | First & second order dynamics, damping, step response metrics |
| 2 | `Exercise_02_Feedback_Control` | Proportional, PI, PID control & disturbance rejection |
| 3 | `Exercise_03_Digital_Control` | Discrete-time PID, actuator saturation, anti-windup, sensor noise |
| 4 | `Exercise_04_Interactive_Controls` | Interactive conveyor belt control exploration |
| 5 | `Exercise_05_Cruise_Control` | Capstone — PID cruise control application |
| 6 | `Exercise_06_Manufacturing_Translation` | Conveyor, oven, and tank — controls meet manufacturing |
| 7 | `Exercise_07_Faults_vs_Disturbances` | Diagnosing faults, disturbances, and sensor issues |
| 8 | `Exercise_08_PdM_and_Control_Decisions` | Control metrics to maintenance decisions and KPIs |